In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q --upgrade transformers accelerate

In [ ]:
import transformers, huggingface_hub

print("transformers :", transformers.__version__)
print("huggingface_hub :", huggingface_hub.__version__)


In [ ]:
import os
import re
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    "    EarlyStoppingCallback,
    ")
    "
    "import warnings
    "warnings.filterwarnings('ignore')


In [ ]:
# ======================
# 1. Paths & config
# ======================
PATH_TRAIN  = "/kaggle/input/nlp-getting-started/train.csv"
PATH_TEST   = "/kaggle/input/nlp-getting-started/test.csv"
PATH_SAMPLE = "/kaggle/input/nlp-getting-started/sample_submission.csv"

    "# Using RoBERTa instead of DistilBERT for better performance
    "MODEL_NAME = \"roberta-base\"

RANDOM_STATE = 42
    "MAX_LEN = 160  # Increased from 128 to capture more context

    "
    "# Set seeds for reproducibility
    "import random
    "def set_seed(seed=42):
    "    random.seed(seed)
    "    np.random.seed(seed)
    "    torch.manual_seed(seed)
    "    torch.cuda.manual_seed_all(seed)
    "    torch.backends.cudnn.deterministic = True
    "    torch.backends.cudnn.benchmark = False
    "
    "set_seed(RANDOM_STATE)
    "
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# ======================
# 2. Load data
# ======================
train_df = pd.read_csv(PATH_TRAIN)
test_df  = pd.read_csv(PATH_TEST)
sample   = pd.read_csv(PATH_SAMPLE)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
    "print("\nTarget distribution:")
    "print(train_df['target'].value_counts())
    "print("\nFirst few rows:")
print(train_df.head())

In [ ]:
# ======================
# 3. Build full text (keyword + location + text)
# ======================
    "# ======================
    "# 3. IMPROVED text preprocessing
    "# ======================
    "def build_text(df):
    '    """Use only text field - often performs better than combining keyword+location"""
    "    return df["text"].fillna("")
    "
    "def clean_text(t: str) -> str:
    '    """Improved text cleaning - less aggressive, preserves important info"""
    "    if pd.isna(t) or t == "":
    "        return ""
    "    
    "    t = str(t).lower()
    "    
    "    # URLs - replace with [URL] token
    "    t = re.sub(r"http\S+|www\.\S+", "[URL]", t)
    "    
    "    # Email addresses
    "    t = re.sub(r"\S+@\S+", "[EMAIL]", t)
    "    
    "    # User mentions - keep as [USER] token
    "    t = re.sub(r"@\w+", "[USER]", t)
    "    
    "    # Hashtags - keep the word without #
    "    t = re.sub(r"#(\w+)", r"\1", t)
    "    
    "    # Keep important punctuation (! ? .) but remove others
    "    t = re.sub(r"[^a-z0-9\s!?."]", " ", t)
    "    
    "    # Multiple spaces
    "    t = re.sub(r"\s+", " ", t).strip()
    "    
    "    return t
    return t

train_df["text_full"] = train_df["text_full"].apply(clean_text)
    "
    "print("Sample cleaned texts:")
    "print(train_df[["text", "text_full", "target"]].head(10))
test_df["text_full"]  = test_df["text_full"].apply(clean_text)

In [ ]:
# ======================
# 4. Split train / validation
# ======================
X_train, X_val, y_train, y_val = train_test_split(
    train_df["text_full"].values,
    train_df["target"].values,
    "    test_size=0.15,  # Reduced from 0.2 for more training data
    random_state=RANDOM_STATE,
    stratify=train_df["target"].values,
)

print("Train size:", len(X_train))
print("Val size  :", len(X_val))    "    "print("\nTrain target distribution:")
    "    "unique, counts = np.unique(y_train, return_counts=True)
    "    "print(dict(zip(unique, counts)))
    "    "print("\nVal target distribution:")
    "    "unique, counts = np.unique(y_val, return_counts=True)
    "    "print(dict(zip(unique, counts)))


In [ ]:
import transformers.utils.hub as hfhub_utils

def _no_templates(*args, **kwargs):
    return []

hfhub_utils.list_repo_templates = _no_templates

In [ ]:
# ======================
    "# ======================
    "# 5. Tokenizer - IMPROVED with dynamic padding
    "# ======================
# ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    "def tokenize_texts(texts, max_length=MAX_LEN):
    return tokenizer(
        list(texts),
    "        padding="max_length",  # Fixed length for efficiency
        truncation=True,
    "        max_length=max_length,
    "        return_tensors=None  # Return lists, not tensors
    )

    "
    "print("Tokenizer loaded. Sample tokenization:")
    "sample_text = "This is a sample tweet about disasters"
    "sample_tokens = tokenizer(sample_text, max_length=MAX_LEN, padding="max_length", truncation=True)
    "print(f"Text: {sample_text}")
    "print(f"Tokens: {tokenizer.convert_ids_to_tokens(sample_tokens['input_ids'])[:10]}")
    "
    "# Tokenize datasets
train_encodings = tokenize_texts(X_train)
val_encodings   = tokenize_texts(X_val)
test_encodings  = tokenize_texts(test_df["text_full"].values)

In [ ]:
# ======================
# 6. Torch Dataset
# ======================
class TweetDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = TweetDataset(train_encodings, y_train)
val_dataset   = TweetDataset(val_encodings, y_val)
test_dataset  = TweetDataset(test_encodings, labels=None)

In [ ]:
# ======================
    "# ======================
    "# 7. Load RoBERTa for classification
    "# ======================
# ======================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    "    num_labels=2,
    "    problem_type="single_label_classification"
)
model.to(device)    "print(f"Model {MODEL_NAME} loaded on {device}")
    "print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# ======================
# 8. Metrics (F1 for Kaggle)
# ======================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    "    f1 = f1_score(labels, preds, average='binary')  # Explicit binary
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [ ]:
# ======================
# 9. TrainingArguments + Trainer
# ======================
training_args = TrainingArguments(
    "# ======================
    "# 9. IMPROVED TrainingArguments + Trainer
    "# ======================
    "training_args = TrainingArguments(
    "    output_dir="./roberta_outputs",
    eval_strategy="epoch",
    save_strategy="epoch",
    "    logging_strategy="steps",
    "    logging_steps=50,
    "    
    "    # Learning rate - slightly lower for RoBERTa
    "    learning_rate=1.5e-5,  # Reduced from 2e-5
    "    
    "    # Batch sizes
    "    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    "    
    "    # Training duration
    "    num_train_epochs=5,  # Reduced from 10, using early stopping instead
    "    
    "    # Regularization
    "    weight_decay=0.01,
    "    warmup_ratio=0.1,  # Warmup for 10% of training steps
    "    
    "    # Optimizations
    "    fp16=True,  # Use mixed precision if GPU available
    "    dataloader_num_workers=2,
    "    
    "    # Model selection
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    "    save_total_limit=2,  # Keep only 2 best checkpoints
    "    
    "    # Logging
    logging_dir="./logs",
    "    report_to="none",  # Disable wandb/tensorboard
    disable_tqdm=False,
    "    
    "    # Reproducibility
    "    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# ======================
# 10. Train
    "print("Starting training...")
# ======================
trainer.train()

# Evaluation finale sur la validation
eval_res = trainer.evaluate()
    "print("\nEvaluating on validation set...")
    "eval_res = trainer.evaluate()
    "print("\nFinal Eval Results:")
    "for key, value in eval_res.items():
    "    print(f"{key}: {value:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
    "import numpy as np
    "import matplotlib.pyplot as plt
    "from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, classification_report

def display_confusion_matrix(y_true, y_pred_logits, dataset_name="Validation", save_path=None):
    """
    Affiche la matrice de confusion + F1 Score pour un dataset.
    Compatible avec les logits (sortie HuggingFace Trainer).
    """

    # logits -> prédictions (classe 0/1)
    if y_pred_logits.ndim > 1:
        y_pred = np.argmax(y_pred_logits, axis=1)
    else:
        y_pred = (y_pred_logits >= 0.5).astype(int)

    # Matrice de confusion
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # F1 via sklearn
    "    f1 = f1_score(y_true, y_pred, average='binary')

    # Affichage
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Not Disaster", "Disaster"]
    )
    disp.plot(cmap=plt.cm.Blues, values_format="d")
    "    plt.title(f"Confusion Matrix on {dataset_name} — F1: {f1:.4f}")

    # Sauvegarde éventuelle
    if save_path is not None:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"✅ Confusion matrix saved to {save_path}")

    plt.show()
    "    
    "    # Classification report
    "    print("\nClassification Report:")
    "    print(classification_report(y_true, y_pred, target_names=["Not Disaster", "Disaster"]))

    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp, "f1": f1}


In [ ]:
# ======================
# Confusion matrix sur la validation
# ======================
    "print("Generating predictions on validation set...")
val_predictions = trainer.predict(val_dataset)
val_logits = val_predictions.predictions   # shape (N_val, 2)
y_true_val = y_val                         # labels que tu as déjà

metrics_val = display_confusion_matrix(
    y_true=y_true_val,
    y_pred_logits=val_logits,
    dataset_name="Validation",
    save_path="confusion_matrix_val.png"   # image sauvegardée
)

    "print("\nValidation Metrics:")
    "print(f"F1 Score: {metrics_val['f1']:.4f}")
    "print(f"True Negatives: {metrics_val['tn']}")
    "print(f"False Positives: {metrics_val['fp']}")
    "print(f"False Negatives: {metrics_val['fn']}")
    "print(f"True Positives: {metrics_val['tp']}")

# ======================
# Prédictions sur test + fichier de submission
# ======================
    "print("\nGenerating test predictions...")
test_predictions = trainer.predict(test_dataset)
test_logits = test_predictions.predictions
test_preds = np.argmax(test_logits, axis=1)

submission = pd.DataFrame({
    "id": test_df["id"],
    "target": test_preds
})

    "submission_filename = "submission_roberta_improved.csv"
    "submission.to_csv(submission_filename, index=False)
    "print(f"✅ {submission_filename} généré !")
    "print(f"\nSubmission preview:")
    "print(submission.head(10))
    "print(f"\nPredictions distribution:")
    "print(submission['target'].value_counts())
